# 11 -- Sound-Tags Facet: Per-Segment AST Tagging + CLAP Text Embedding (Stretch tier)

Adds a **7th similarity facet**, `sound_tags`: per-segment (~5s window, same granularity as
every other facet) AST-detected sound/instrument tags, embedded via **CLAP's text encoder**
(not its audio encoder) into the same 512-dim joint space `sound`'s CLAP-audio embeddings
already live in. Reuses the real `sonic_explorer` package throughout -- no notebook-local
reimplementation:

- `sonic_explorer/facets/tags.py` -- `SoundTagsFacet`, the new two-stage facet (AST tag ->
  CLAP text embed, both inside one `embed(audio, sr)` call so it satisfies the same interface
  every other facet does).
- `sonic_explorer/pipeline/embed_tags.py` -- `run_batch_tags_embedding`, the batch job this
  notebook actually runs. Same compute-once/checkpoint/per-song-failure-isolation discipline
  as `embed_stems.py`.
- Already registered in `sonic_explorer/facets/registry.py` -- once this notebook's output is
  synced back to the repo, Explore/Moment Matcher/Ask the DJ pick it up automatically, no
  further code changes needed (same pattern every facet here has followed).

## Why per-segment, not song-level (a real design decision, not the default)

AST sound tags already exist **once per song** (`songs.sound_tags`, a single middle-10s
slice -- see Methodology 7b / `scripts/generate_song_descriptions.py`). The cheap option would
have been embedding that existing per-song text once and reusing it across a song's segments.
**Deliberately not what this notebook does** -- per-segment tagging was chosen specifically
because it genuinely differentiates *within* a song, the same way every other facet does. Real
evidence from smoke-testing this before writing the batch job (song: "3rd Chair," a cello/violin
instrumental): three consecutive 5s windows returned genuinely different top tags --
`Violin, fiddle > Cello` in one window, `Cello > Violin, fiddle` in the next, `Double bass`
appearing only in a third -- real moment-to-moment variation a single song-level tag could never
capture. A song-level-only version would have been faster to build but couldn't have supported
that.

## Honest time cost -- read before running Section 5

This is **not** a light job. Real, measured numbers from testing this pipeline locally (not
guessed): AST tagging averages **~2.49s/segment on CPU**, and this library has **14,602
segments** across 1,400 songs -- **~10.1 hours of pure inference**, plus roughly another **~1.2
hours** of audio-loading overhead (1,400 songs x ~3s each) = **~10.5-11.5 hours sequential, on
CPU.**

**Multiprocessing was tested and made this *worse*, not better** -- 4 parallel CPU workers
measured at 3.14s/segment (real, wall-clock), slower than one process alone (2.49s/segment),
almost certainly CPU/memory-bandwidth contention between concurrent transformer forward passes.
This notebook does not attempt parallelism as a result.

**GPU support has been added** (`pipeline/sound_tagging.py`'s tagger now requests `device=0`
when `torch.cuda.is_available()`, previously hardcoded to CPU) as a real, low-risk potential
speedup if you run this with Runtime -> Change runtime type -> GPU. This is a plausible,
well-established expectation for a transformer this size, **not an empirically verified number**
-- there's no GPU available in the environment this notebook was written/tested in. Section 4's
smoke test measures your *actual* per-segment rate on whatever runtime you choose, live, before
you commit to the full run in Section 5 -- read that number before proceeding, don't assume
either the CPU or GPU estimate applies to you specifically.

## Requires the `[colab]` extra (transformers + torch), no GPU strictly required

Runtime -> Change runtime type -> GPU is optional but likely to matter a lot here specifically
(unlike the sound_tags text-embedding step itself, which is lightweight either way) -- see the
time-cost section above.

## Output (in `MyDrive/SonicExplorer/artifacts/`)
- `sound_tags.index` -- FAISS index, one 512-dim vector per successfully-tagged segment
- `sonic_explorer.db` -- same DB as before, now with `embedding_status` rows for `sound_tags`
  too (`done` for tagged segments, `skipped` for segments where AST returned nothing but
  generic "Music"/"Musical instrument" tags -- rare in practice, see Section 3's real numbers)

## 1. Clone the repo and install sonic_explorer (with the `[colab]` extra)

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[colab]'])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print('sonic_explorer installed from', REPO_DIR)
print('GPU available:', torch.cuda.is_available(), '--', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Mount Drive and set up paths

Same Drive-mounted DB + curated-audio folder every other notebook uses -- the library and its
sound/harmony/stem embeddings should already be there from notebooks 01-03.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
CURATED_DIR = DRIVE_ROOT / 'fma_curated'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = ARTIFACTS_DIR / 'sonic_explorer.db'

print('Curated audio:', CURATED_DIR)
print('Artifacts (DB + indexes):', ARTIFACTS_DIR)

## 3. Load repositories + the new facet + build the manifest

Same manifest source as notebooks 02/03 -- `curated_tracks.csv`'s `relative_path` column.
`run_batch_tags_embedding` (like `run_batch_stem_embedding`) never creates songs itself, only
adds `sound_tags` vectors for songs that already exist.

In [ ]:
import pandas as pd

from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository
from sonic_explorer.repository.embedding_repository import EmbeddingRepository
from sonic_explorer.facets.tags import SoundTagsFacet, NoTagsDetected, tags_to_text
from sonic_explorer.pipeline.sound_tagging import AST_SAMPLE_RATE, get_descriptive_tags

conn = init_db(DB_PATH)
song_repo = SongRepository(conn)
embedding_repo = EmbeddingRepository(conn, artifacts_dir=ARTIFACTS_DIR)

facet = SoundTagsFacet()
embedding_repo.load_index('sound_tags')
print('Resuming sound_tags index size:', embedding_repo.index_size('sound_tags'))

manifest = pd.read_csv(CURATED_DIR / 'curated_tracks.csv')
print(f'{len(manifest)} curated tracks')

## 4. Smoke test -- tag + embed a few real segments, and measure YOUR real rate

Do this before the full run. Two things to check:
1. **Are the tags genuinely descriptive and do they vary within a song?** (the whole reason
   this is per-segment, not song-level -- see the notebook intro).
2. **What's your actual per-segment rate on this runtime?** The ~10.5-11.5h estimate above is
   measured on CPU with no GPU available. If you're on a GPU runtime, this cell tells you your
   real number instead of a guess -- extrapolate Section 4b's printed estimate before deciding
   whether to run Section 5 as-is, overnight, or not at all right now.

In [ ]:
import time

existing_mask = manifest['relative_path'].apply(lambda p: (CURATED_DIR / p).exists())
existing_manifest = manifest[existing_mask]
print(f'{len(existing_manifest)} of {len(manifest)} curated tracks have audio present in this Colab session')

sample_row = existing_manifest.sample(1, random_state=42).iloc[0]
sample_song = song_repo.get_song_by_fma_track_id(int(sample_row.track_id))
print(f'Smoke-testing on: "{sample_song.title}" by {sample_song.artist}\n')

import librosa
audio, sr = librosa.load(str(CURATED_DIR / sample_row.relative_path), sr=AST_SAMPLE_RATE, mono=True)
segments = song_repo.get_segments(sample_song.id)[:4]

times = []
for seg in segments:
    start, end = int(seg.start_sec * sr), int(seg.end_sec * sr)
    window = audio[start:end]

    t0 = time.time()
    raw_tags = get_descriptive_tags(window, sr)
    elapsed = time.time() - t0
    times.append(elapsed)

    text = tags_to_text(raw_tags)
    print(f'[{seg.start_sec:.1f}-{seg.end_sec:.1f}s] ({elapsed:.2f}s) raw tags: {raw_tags}')
    print(f'  -> embedded text: {text!r}')

print(f'\nMean per-segment tagging time on this runtime: {sum(times) / len(times):.2f}s')

### 4b. Extrapolate to the full library, using YOUR measured rate

In [ ]:
TOTAL_SEGMENTS = song_repo.count_segments()
TOTAL_SONGS = len(song_repo.list_songs())

mean_rate = sum(times) / len(times)
est_tagging_hours = TOTAL_SEGMENTS * mean_rate / 3600
est_loading_hours = TOTAL_SONGS * 3.0 / 3600  # ~3s/song audio load, matching the CPU measurement this notebook cites

print(f'{TOTAL_SEGMENTS} total segments across {TOTAL_SONGS} songs in this library')
print(f'Measured rate on this runtime: {mean_rate:.2f}s/segment')
print(f'Estimated full-run time: ~{est_tagging_hours:.1f}h tagging + ~{est_loading_hours:.1f}h audio loading '
      f'= ~{est_tagging_hours + est_loading_hours:.1f}h total')
print('\nCompare against this notebook\'s own CPU baseline: ~10.1h tagging + ~1.2h loading = ~11.3h total.')

## 5. Run the batch pipeline

**Read Section 4b's printed estimate for YOUR runtime before running this cell.** Same
checkpointing discipline as every other batch job here: a segment is only ever marked `'done'`
after its vector is durably saved to the FAISS index on disk, and a single song's failure is
isolated (reported, not fatal) -- one bad file shouldn't lose progress on the other 1399.
Safely resumable if this Colab session disconnects partway through (a real risk at this
runtime, not a hypothetical) -- just re-run this cell; already-finished segments are skipped,
not re-tagged.

In [ ]:
from sonic_explorer.pipeline.embed_tags import run_batch_tags_embedding

_run_start = time.time()
_segments_at_start = embedding_repo.index_size('sound_tags')

def log_progress(done, total):
    elapsed_h = (time.time() - _run_start) / 3600
    current_size = embedding_repo.index_size('sound_tags')
    new_segments = current_size - _segments_at_start
    rate = new_segments / elapsed_h if elapsed_h > 0 else 0
    remaining = TOTAL_SONGS - done
    print(f'[{done}/{total} songs] {elapsed_h:.2f}h elapsed -- sound_tags index size: {current_size} '
          f'(+{new_segments} this run, ~{rate:.0f} segments/h) -- {remaining} songs left')

def log_error(track_id, exc):
    print(f'  WARNING: track {track_id} failed ({type(exc).__name__}: {exc}) -- skipped, will retry next run')

failed = run_batch_tags_embedding(
    manifest=manifest,
    audio_dir=CURATED_DIR,
    song_repo=song_repo,
    embedding_repo=embedding_repo,
    facet=facet,
    checkpoint_every=25,
    on_checkpoint=log_progress,
    on_error=log_error,
)

print('\nBatch sound_tags embedding complete.')
print('Final sound_tags index size:', embedding_repo.index_size('sound_tags'))
if failed:
    print(f'{len(failed)} track(s) failed and were skipped: {failed}')

## 6. Sanity check -- does the facet actually return something sensible?

Same pattern as notebooks 02/03's own checks: pull a segment's `sound_tags` vector back out (no
re-tagging needed) and look at its nearest neighbors in other songs, excluding the query's own
song.

In [ ]:
import random

sample_song = random.choice(song_repo.list_songs())
segments = song_repo.get_segments(sample_song.id)
query_seg = segments[len(segments) // 2] if segments else None

if query_seg is not None and embedding_repo.status(query_seg.id, 'sound_tags') == 'done':
    query_vec = embedding_repo.get_vector('sound_tags', query_seg.id)
    raw_matches = embedding_repo.search('sound_tags', query_vec, k=20)
    other_song_matches = [
        (seg_id, score) for seg_id, score in raw_matches
        if song_repo.get_segment(seg_id).song_id != sample_song.id
    ][:6]

    print(f'Query (sound_tags facet): "{sample_song.title}" by {sample_song.artist} '
          f'[{query_seg.start_sec:.1f}-{query_seg.end_sec:.1f}s]\n')
    for seg_id, score in other_song_matches:
        seg = song_repo.get_segment(seg_id)
        match_song = song_repo.get_song(seg.song_id)
        print(f'  {score:.3f}  "{match_song.title}" by {match_song.artist} ({match_song.genre_top}) '
              f'[{seg.start_sec:.1f}-{seg.end_sec:.1f}s]')
else:
    print(f'"{sample_song.title}" has no sound_tags vector yet -- pick another sample or re-run the batch cell.')

## Done -- and what's still manual after this

`MyDrive/SonicExplorer/artifacts/` now has `sound_tags.index` alongside the other six. From
here, matching how every earlier facet's Colab output made it into the live app:

1. **Sync `sonic_explorer.db` and `sound_tags.index` down to this repo's local
   `data/artifacts/`**, the same way notebook 02/03's output was synced.
2. **Re-run `scripts/build_deploy_subset.py`** -- it already reads whatever facets are
   registered (`default_registry().names()`), so this picks up `sound_tags` automatically and
   copies its vectors into the small, committed `deploy_data/` Streamlit Community Cloud
   actually runs against. No code change needed for this step.
3. **Run genre-cohesion evaluation for real** (`scripts/run_evaluation.py` or the equivalent
   `genre_cohesion_at_k(..., facet_name='sound_tags')` call) -- this facet is registered and
   will show up in Explore/Moment Matcher's facet picker immediately, but its actual
   evaluation number (is it a *useful* signal, not just a working one) still needs a real run
   against the populated index, the same check every other facet got.
4. **Update the "six facets" copy** -- `streamlit_app/pages/1_Methodology.py` and `2_Results.py`
   both hardcode `FACET_ORDER = ["sound", "harmony", "vocal", "drums", "bass", "instrumental"]`
   and several headers literally say "six similarity facets" -- these need a real edit once
   `sound_tags` has real data worth reporting, not before (an empty/untested facet showing up
   in a "seven facets, evaluated" narrative would be worse than the current honest gap).

None of these four steps happen automatically just by running this notebook -- consistent with
this project's own established discipline (see `CLAUDE.md`): a script produces a real artifact,
and wiring it into what's actually shown/shipped is always a distinct, deliberate step.